# 모델 학습

In [1]:
import os
import yaml

yaml_path = "주류탐지_데이터셋/data.yaml"

# 1. 기존 yaml 파일 읽기
with open(yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

# 2. 현재 작업 폴더(OSS_Final_Team_Project) 기준 절대경로 지정
base_dir = os.path.abspath("주류탐지_데이터셋")
data["path"] = base_dir
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

# 3. 보정된 설정 저장
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data, f, allow_unicode=True)

print("✅ data.yaml 경로 설정 완료!")
print(f"📦 클래스 수: {data.get('nc')}")
print(f"🏷️ 클래스 이름: {data.get('names')}")


✅ data.yaml 경로 설정 완료!
📦 클래스 수: 3
🏷️ 클래스 이름: {0: 'beer-bottle', 1: 'beer-can', 2: 'soju-bottle'}


In [ ]:
from ultralytics import YOLO

# 1. 폴더에 있는 small 모델 가중치 불러오기
model = YOLO("yolov8s.pt")

# 2. 커스텀 데이터셋 학습 시작
results = model.train(
    data="주류탐지_데이터셋/data.yaml",
    epochs=50,                  # 기본 50 에포크 권장
    imgsz=640,                  # 입력 이미지 해상도
    batch=16,                   # 메모리 부족(OOM) 발생 시 8로 조정
    device="mps",               # Mac Metal GPU 가속 (Apple Silicon)
    name="drink_detector_s",    # 학습 결과 저장 폴더 이름
    save=True,
    plots=True
)


Ultralytics 8.4.210 🚀 Python-3.14.6 torch-2.15.0 MPS (Apple M2)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=주류탐지_데이터셋/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drink_detector_s, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overla

Matplotlib is building the font cache; this may take a moment.


Overriding model.yaml nc=80 with nc=3

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     29056  ultralytics.nn.modules.block.C2f             [64, 64, 1, True]             
  3                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  4                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  5                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  6                  -1  2    788480  ultralytics.nn.modules.block.C2f             [256, 256, 2, True]           
  7                  -1  1   1180672  ultralytics

/opt/anaconda3/lib/python3.14/site-packages/torch/autograd/graph.py:1113: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:224.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/opt/anaconda3/lib/python3.14/site-packages/torch/autograd/graph.py:1113: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/

       1/50      7.38G     0.9424      1.665      1.167          8        640: 100% ━━━━━━━━━━━━ 59/59 161.8s/it 2:39:094ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% ━━━━━━────── 2/4 215.2s/it 3:38<7:104WARNING ⚠️ NMS time limit 3.600s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 89.1s/it 5:56<2:31
                   all        112        321      0.415      0.387      0.322      0.234

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


In [ ]:
# 가장 잘 학습된 가중치 불러오기
best_model = YOLO("runs/detect/drink_detector_s/weights/best.pt")

# test 폴더 내 이미지 한 장으로 추론 테스트
test_results = best_model.predict(
    source="주류탐지_데이터셋/test/images",
    save=True,
    conf=0.5
)

print("✅ 테스트 완료! runs/detect/predict 폴더에서 감지된 사진을 확인할 수 있습니다.")
